<a href="https://colab.research.google.com/github/Biteo326/blank-app/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the required Libraries
!pip install deep-translator vaderSentiment nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.0 MB/s eta 0:00:00


In [2]:
# Import libraries
import pandas as pd
import re
import nltk

from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
# Load Dataset
from google.colab import files

uploaded = files.upload()

df = pd.read_csv('/content/Dataset.csv') #Change the directory if need

print("Dataset Shape:", df.shape)
df.head()

Saving Dataset.csv to Dataset.csv
Dataset Shape: (3006, 19)


,User Id,User Shop Id,User Name,Anonymous,Comment Id,Comment Date,Comment,Rating,Detail Rating,Comment Images,Comment Videos,Bought Products,Product Id,Product Url,Product Name,Product Image,Shop Id,Region,Scraped At
0,123154054,NaN,n*****5,Yes,58837403987,2025-07-10T15:27:00.000Z,Performance:Best\nQuality:Great\n\nTq seller P...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-bs-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-28T04:55:59.116Z
1,59528895,NaN,mninh95,No,17258222843,2024-10-19T10:34:45.000Z,Quality:good\nPerformance:good\n\nI am really ...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-ws-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-28T04:55:59.116Z
2,231187758,NaN,michellelin95,No,16606262072,2024-09-16T17:37:50.000Z,Performance:Good\nQuality:Good\n\nDapat beli m...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-ws-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-28T04:55:59.116Z
3,1302584326,NaN,n*****8,Yes,27305922538,2025-01-09T02:38:51.000Z,Quality:Good\nPerformance:Nicee\n\nAirpods tha...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-ws-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-28T04:55:59.116Z
4,1411611864,NaN,_norazl1nn,No,30268997432,2025-02-06T06:48:56.000Z,Quality:very good!\nPerformance:also good\n\nF...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-ws-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-28T04:55:59.116Z


In [4]:
# Check Column Names
print(df.columns)

Index(['User Id', 'User Shop Id', 'User Name', 'Anonymous', 'Comment Id',
       'Comment Date', 'Comment', 'Rating', 'Detail Rating', 'Comment Images',
       'Comment Videos', 'Bought Products', 'Product Id', 'Product Url',
       'Product Name', 'Product Image', 'Shop Id', 'Region', 'Scraped At'],
      dtype='object')


In [5]:
# Keep only the columns of Comments
df = df[['Comment']]

print("Rows before cleaning:", len(df))
df.head()

Rows before cleaning: 3006


,Comment
0,Performance:Best\nQuality:Great\n\nTq seller P...
1,Quality:good\nPerformance:good\n\nI am really ...
2,Performance:Good\nQuality:Good\n\nDapat beli m...
3,Quality:Good\nPerformance:Nicee\n\nAirpods tha...
4,Quality:very good!\nPerformance:also good\n\nF...


In [6]:
# Remove Empty Comments

df = df.dropna(subset=['Comment'])

df = df[
    df['Comment']
    .astype(str)
    .str.strip() != ''
]

print("Remaining rows:", len(df))

Remaining rows: 3006


In [7]:
# Text Cleaning Function

def clean_text(text):

    text = str(text).lower()

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove emojis
    text = re.sub(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "]+",
        "",
        text,
        flags=re.UNICODE
    )

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [8]:
# Apply Cleaning

df['Cleaned_Comment'] = df['Comment'].apply(clean_text)

df = df[df['Cleaned_Comment'] != '']

print("Rows after text cleaning:", len(df))

Rows after text cleaning: 3006


In [9]:
#Remove Empty Cleaned Reviews

df = df[
    df['Cleaned_Comment'] != ''
]

print(df.shape)

(3006, 2)


In [10]:
# Translate the languages to English

def translate_text(text):

    try:
        return GoogleTranslator(
            source='auto',
            target='en'
        ).translate(text)

    except:
        return text

In [11]:
# Apply the translator

from tqdm import tqdm

tqdm.pandas()

df['English_Comment'] = (
    df['Cleaned_Comment']
    .progress_apply(translate_text)
)

100%|██████████| 3006/3006 [24:16<00:00,  2.06it/s]


In [12]:
#Tokenization

from nltk.tokenize import word_tokenize

df['Tokens'] = (
    df['English_Comment']
    .apply(word_tokenize)
)

In [13]:
#Remove Stopwords

from nltk.corpus import stopwords

stop_words = set(
    stopwords.words('english')
)

df['Tokens'] = (
    df['Tokens']
    .apply(
        lambda words:
        [w for w in words
         if w not in stop_words]
    )
)

In [14]:
#Stemming

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

df['Stemmed'] = (
    df['Tokens']
    .apply(
        lambda words:
        [stemmer.stem(w)
         for w in words]
    )
)

In [15]:
#Convert Back to Sentence

df['Processed_Text'] = (
    df['Stemmed']
    .apply(lambda x: ' '.join(x))
)

In [16]:
# Sentiment Analysis

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):

    score = analyzer.polarity_scores(text)['compound']

    if score >= 0.10:
        return 'Positive'

    elif score <= -0.10:
        return 'Negative'

    else:
        return 'Neutral'

In [17]:
# Generate Sentiment Labels

df['Sentiment'] = df['English_Comment'].apply(
    get_sentiment
)

In [18]:
# View Sample Results

df[['Comment',
    'Cleaned_Comment',
    'English_Comment',
    'Sentiment']].head(20)

,Comment,Cleaned_Comment,English_Comment,Sentiment
0,Performance:Best\nQuality:Great\n\nTq seller P...,performance best quality great tq seller produ...,"performance, best quality, great tq seller, ch...",Positive
1,Quality:good\nPerformance:good\n\nI am really ...,quality good performance good i am really in l...,quality good performance good i am really in l...,Positive
2,Performance:Good\nQuality:Good\n\nDapat beli m...,performance good quality good dapat beli murah...,"good performance, good quality, you can buy it...",Positive
3,Quality:Good\nPerformance:Nicee\n\nAirpods tha...,quality good performance nicee airpods that ha...,quality good performance nicee airpods that ha...,Positive
4,Quality:very good!\nPerformance:also good\n\nF...,quality very good performance also good fast d...,quality very good performance also good fast d...,Positive
5,Performance:very satisfied\nQuality:100% Perfe...,performance very satisfied quality perfect fas...,performance very satisfied quality perfect fas...,Positive
6,Performance:good\nQuality:good\n\nDh test sala...,performance good quality good dh test salah yg...,performance good quality good dh test the one ...,Positive
7,Quality:sangat padu\nPerformance:bass mantap g...,quality sangat padu performance bass mantap gi...,"very solid quality, solid bass performance, un...",Positive
8,Performance:Good\n\nGood quality walaupun harg...,performance good good quality walaupun harga m...,performance good good quality even though the ...,Positive
9,Performance:best\nQuality:good product\n\nThan...,performance best quality good product thank yo...,"performance, best quality, good product, thank...",Positive


In [19]:
# Check Sentiment Distribution

print(df['Sentiment'].value_counts())

Sentiment
Positive    2593
Negative     264
Neutral      149
Name: count, dtype: int64


In [20]:
# Save Cleaned Dataset

df.to_csv(
    'Final_Cleaned_Dataset.csv',
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [21]:
# Download Cleaned Dataset

from google.colab import files

files.download('Final_Cleaned_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>